In [1]:
import pandas as pd
import re
from difflib import get_close_matches
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.cross_decomposition import PLSRegression
from sklearn.feature_selection import VarianceThreshold
import numpy as np

In [2]:
# Read the genetic data
genetic_data = pd.read_csv(r"C:\Users\fatemehm\OneDrive - Royal HZPC Group\Desktop\internship\bio_rep1\GeneMarkers-95 varieties 1.csv")

genetic_data.info
genetic_data.head(10)

,taglo_id,ATLANTIC_124776,ESCORT_159228,ARGOS_120931,ALTURAS_329540,ELMUNDO_835520,NADINE_241950,VIOLETQUEEN_3345402,DEODARA_513721,MEMPHIS_2279529,...,DIAMANT_153502,INNOVATOR_234757,TRIPLE7_3347283,SPUNTA_283648,ANTI_118190,CARRERA_142554,FORTUS_3279866,SMART_1883446,SALINERO_253609,FABULA_171173
0,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,10,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,13,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,17,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,21,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,22,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,39,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,48,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,49,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,54,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [3]:
genetic_data.isnull().sum()

taglo_id           0
ATLANTIC_124776    0
ESCORT_159228      0
ARGOS_120931       0
ALTURAS_329540     0
                  ..
CARRERA_142554     0
FORTUS_3279866     0
SMART_1883446      0
SALINERO_253609    0
FABULA_171173      0
Length: 95, dtype: int64

In [4]:
# Clean column names
genetic_columns_clean = [c.strip().upper().replace(" ", "").replace(".", "") for c in genetic_data.columns]

# Assign cleaned names back to DataFrame
genetic_data.columns = genetic_columns_clean

# Now you can view columns
print(genetic_data.columns)


Index(['TAGLO_ID', 'ATLANTIC_124776', 'ESCORT_159228', 'ARGOS_120931',
       'ALTURAS_329540', 'ELMUNDO_835520', 'NADINE_241950',
       'VIOLETQUEEN_3345402', 'DEODARA_513721', 'MEMPHIS_2279529',
       'AGRIA_125336', 'PAYETTERUSSET_4617676', 'IVORYRUSSET_1360551',
       'SABABA_4959615', 'CECILE_147140', 'FRISIA_163592', 'CARDYMA_3345873',
       'RICKEYRUSSET_4640884', 'ANIVIA_4446779', 'AVARNA_1630052',
       'DONALD_158741', 'JAERLA_231308', 'ARIZONA_3056017',
       'SARPOMIRA_1629377', 'MARILYN_1632173', 'DESIREE_139717',
       'MUSE_6159099', 'ADORA_123802', 'TAURUS_1883495', 'BERBER_121293',
       'RANGERRUSSET_265488', 'BINTJE_126680', 'FENWAYRED_4055844',
       'KONDOR_248484', 'SAGITTA_1459411', 'CLEARWATERR_2721777',
       'NICOLA_248062', 'TETONRUSSET_3029162', 'FESTIEN_172619',
       'HANSA_180216', 'PEEWEERUSSET_3347291', 'PRINCEOFORANGE_416115',
       'SUPERIOR_1477892', 'VANGOGH_294181', 'CHALLENGER_1883438',
       'JENNIFER_3221728', 'PARELLA_2983716', 'CH

In [ ]:
merged_flavor_sensory= pd.read_csv(r'C:\Users\fatemehm\OneDrive - Royal HZPC Group\Desktop\internship\bio_rep1\aroma_plus_predicted_flavors_94.csv')
merged_flavor_sensory

,Variety,"1-Nonanol (Floral, waxy)","2,3-Butanedione (Butter, creamy, sweet)","Benzaldehyde (Almond, cherry, fruity)","Butanal, 3-methyl- (Fruity, malty, chocolate)","Cyclotrisiloxane, hexamethyl- (Chemical, silicone-like)","Decanal (Citrus, floral)","Dimethyl trisulfide (Garlic, onion, sulfurous)","Furfural (Sweet, almond, caramel)","Hexanal (Green, grassy)",...,Bitter Flavour__pred_lin,Earthy Flavour__pred_lin,Sour Flavour__pred_lin,Fresh Flavour__pred_lin,Sweet Flavour__pred_lin,Root/ Vegetable Flavour__pred_lin,Farmyard (grass/hay) flavour__pred_lin,Bitter Aftertaste__pred_lin,Sour Aftertaste__pred_lin,Sweet Aftertaste__pred_lin
0,ADORA,0.0,0.0,17.984870,15.836435,17.417208,18.203338,0.000000,0.0,0.000000,...,7.862255,17.459044,5.523512,31.808752,4.853171,13.197864,13.838047,8.067712,7.835167,3.084403
1,AGRIA,0.0,0.0,14.859493,14.154847,15.947810,14.499810,17.210770,0.0,0.000000,...,7.596909,17.734322,6.660871,31.404579,4.903883,12.447347,14.178856,7.906308,6.796900,3.033925
2,ALOUETTE,0.0,0.0,15.205705,15.262178,14.678352,17.462371,0.000000,0.0,0.000000,...,7.657185,17.033344,6.400143,31.258695,4.471912,12.709900,14.215778,7.829244,7.199740,3.117696
3,ALTHEA,0.0,0.0,19.535081,14.984406,17.704453,21.158234,0.000000,0.0,0.000000,...,7.658574,17.841248,6.292800,32.076281,5.492117,12.370690,14.063781,8.020408,7.248337,3.057748
4,ALTURAS,0.0,0.0,20.293776,14.482897,14.257052,16.590508,0.000000,0.0,17.245720,...,7.464395,16.805916,6.901308,31.322823,4.704826,12.287131,14.324117,7.627450,7.059270,3.270196
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,TETON RUSSET,0.0,0.0,14.711181,14.280807,16.957599,13.224951,0.000000,0.0,14.538894,...,7.578729,16.960024,6.627112,31.308889,4.595239,12.759721,14.242103,7.706123,7.075521,3.194071
90,TRIPLE7,0.0,0.0,13.650265,0.000000,13.360218,14.801468,0.000000,0.0,0.000000,...,7.560665,16.884959,7.087250,31.328892,4.559579,12.406272,14.398713,7.655090,6.842907,3.223438
91,VAN GOGH,0.0,0.0,17.966916,15.946913,17.526481,14.548161,19.014337,0.0,0.000000,...,7.678540,17.274197,6.074573,31.448654,4.654003,12.784897,14.048397,7.928240,7.390274,3.098513
92,VE 71-105,0.0,0.0,17.250436,15.551781,16.912431,14.262653,19.086670,0.0,0.000000,...,7.853380,17.491083,5.520243,31.568516,4.616523,13.281992,13.753395,8.057515,7.805345,3.140601


In [15]:
import os, re
import pandas as pd

# ========= 1) Variety normalization & corrections =========
def normalize_variety(name):
    if not isinstance(name, str):
        return ""
    name = name.upper().replace(".", "").replace(" ", "")
    name = re.sub(r'\bR\b', '', name)            # drop standalone 'R'
    name = re.sub(r'RUSSET', '', name)           # drop 'RUSSET'
    name = name.strip().split('_')[0]            # drop trailing _... tokens
    return name

name_corrections = {
    'CLEARWATERR': 'CLEARWATER',
    'ALVERSTONER': 'ALVERSTONE'
}

# ========= 2) Fix genetic table so TAGLO IDs are rows (varieties) and markers are columns =========
gd = genetic_data.copy()  # <- your raw genetic dataframe
gd.columns = gd.columns.map(str)

def is_numeric_id(s): 
    s = str(s)
    return s.isdigit()

frac_numeric_cols = sum(is_numeric_id(c) for c in gd.columns) / max(1, len(gd.columns))

if 'TAGLO_ID' in gd.columns and frac_numeric_cols < 0.5:
    # Likely TAGLO_ID holds *variety IDs* as a column; markers are rows → transpose
    gd = gd.set_index('TAGLO_ID').T
elif 'TAGLO_ID' in gd.columns and frac_numeric_cols >= 0.5:
    # TAGLO_ID is the variety key; markers already columns
    gd = gd.set_index('TAGLO_ID')
# else: assume current shape already has varieties on index

# Normalize & correct index
gd.index = pd.Index([normalize_variety(ix) for ix in gd.index])
gd.index = gd.index.to_series().replace(name_corrections).astype(str)
gd.index.name = 'Variety'

# Coerce all markers to numeric
gd = gd.apply(pd.to_numeric, errors='coerce')

# Collapse duplicate varieties by mean
gd = gd.groupby(level=0).mean(numeric_only=True)

# Prefix marker columns
gd.columns = ["GEN__" + str(c) for c in gd.columns]

print("Genetic table after fix:", gd.shape)
print("Sample genetic columns:", list(gd.columns[:5]))

# ========= 3) Keep only predicted flavor columns and merge =========
# merged_flavor_sensory already loaded from your CSV
flavor_df = merged_flavor_sensory.copy()

# Keep Variety + only predicted columns (__pred, __pred_cal, __pred_lin)
pred_mask = flavor_df.columns.str.contains(r'__(pred|pred_cal|pred_lin)$', case=False, regex=True)
flavor_keep = flavor_df.loc[:, ['Variety'] + list(flavor_df.columns[pred_mask])]
# Normalize key the same way as genetics
flavor_keep['Variety'] = flavor_keep['Variety'].map(normalize_variety)
flavor_keep['Variety'] = flavor_keep['Variety'].replace(name_corrections)

# Set index to Variety for join
F = flavor_keep.dropna(subset=['Variety']).set_index('Variety')
F.index.name = 'Variety'

# Inner join on normalized Variety
merged_genetic_flavor = gd.join(F, how='inner')

print("Flavor-preds shape:", F.shape)
print("Merged shape:", merged_genetic_flavor.shape)
print("Missing in genetic (from flavor):", sorted(set(F.index) - set(gd.index)))
print("Missing in flavor (from genetic):", sorted(set(gd.index) - set(F.index)))

# ========= 4) Save =========
OUTPUT_DIR = r"C:\Users\fatemehm\OneDrive - Royal HZPC Group\Desktop\internship\bio_rep1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_path = os.path.join(OUTPUT_DIR, "genetic_plus_predicted_flavors_94.csv")
parquet_path = os.path.join(OUTPUT_DIR, "genetic_plus_predicted_flavors_94.parquet")

merged_genetic_flavor.to_csv(csv_path)
print("Saved CSV:", csv_path)

try:
    merged_genetic_flavor.to_parquet(parquet_path, index=True)
    print("Saved Parquet:", parquet_path)
except Exception as e:
    print("Parquet not written:", e)
merged_genetic_flavor

Genetic table after fix: (94, 262434)
Sample genetic columns: ['GEN__7', 'GEN__10', 'GEN__13', 'GEN__17', 'GEN__21']


C:\Users\fatemehm\AppData\Local\Temp\ipykernel_18996\4258635809.py:59: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  pred_mask = flavor_df.columns.str.contains(r'__(pred|pred_cal|pred_lin)$', case=False, regex=True)


Flavor-preds shape: (94, 33)
Merged shape: (94, 262467)
Missing in genetic (from flavor): []
Missing in flavor (from genetic): []
Saved CSV: C:\Users\fatemehm\OneDrive - Royal HZPC Group\Desktop\internship\bio_rep1\genetic_plus_predicted_flavors_94.csv
Parquet not written: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.


,GEN__7,GEN__10,GEN__13,GEN__17,GEN__21,GEN__22,GEN__39,GEN__48,GEN__49,GEN__54,...,Bitter Flavour__pred_lin,Earthy Flavour__pred_lin,Sour Flavour__pred_lin,Fresh Flavour__pred_lin,Sweet Flavour__pred_lin,Root/ Vegetable Flavour__pred_lin,Farmyard (grass/hay) flavour__pred_lin,Bitter Aftertaste__pred_lin,Sour Aftertaste__pred_lin,Sweet Aftertaste__pred_lin
Variety,,,,,,,,,,,,,,,,,,,,,
ADORA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,7.862255,17.459044,5.523512,31.808752,4.853171,13.197864,13.838047,8.067712,7.835167,3.084403
AGRIA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,7.596909,17.734322,6.660871,31.404579,4.903883,12.447347,14.178856,7.906308,6.796900,3.033925
ALOUETTE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,7.657185,17.033344,6.400143,31.258695,4.471912,12.709900,14.215778,7.829244,7.199740,3.117696
ALTHEA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,7.658574,17.841248,6.292800,32.076281,5.492117,12.370690,14.063781,8.020408,7.248337,3.057748
ALTURAS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,7.464395,16.805916,6.901308,31.322823,4.704826,12.287131,14.324117,7.627450,7.059270,3.270196
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TETON,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,7.578729,16.960024,6.627112,31.308889,4.595239,12.759721,14.242103,7.706123,7.075521,3.194071
TRIPLE7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,7.560665,16.884959,7.087250,31.328892,4.559579,12.406272,14.398713,7.655090,6.842907,3.223438
VANGOGH,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,7.678540,17.274197,6.074573,31.448654,4.654003,12.784897,14.048397,7.928240,7.390274,3.098513


# Feature Reduction on X (Genetic Markers)

Variance thresholding (remove near-constant features)

In [8]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold

# --- Column groups (assumes prefixes already set) ---
combined_df.columns = combined_df.columns.map(str)
TARGETS    = [c for c in combined_df.columns if c.startswith("TARGET__")]
GEN_COLS   = [c for c in combined_df.columns if c.startswith("GEN__")]
AROMA_COLS = [c for c in combined_df.columns if c.startswith("AROMA__")]

print(f"Targets: {len(TARGETS)} | GEN cols: {len(GEN_COLS)} | AROMA cols: {len(AROMA_COLS)}")

# --- FAST: identify non-numeric only once and coerce just those ---
feat_cols = GEN_COLS + AROMA_COLS
non_num = combined_df[feat_cols].columns[combined_df[feat_cols].dtypes.eq("object")].tolist()
if non_num:
    combined_df[non_num] = combined_df[non_num].apply(pd.to_numeric, errors="coerce")

# (Optional) Downcast a numeric *view* to float32 only when forming numpy arrays later.

# ===== GENETICS: ultra-fast variance filter =====
# Optional: hard cap number of columns by random subsample *before* any math (speeds everything)
max_gen_prefilter = 30000   # set None to disable; try 10k–50k depending on your machine
if max_gen_prefilter and len(GEN_COLS) > max_gen_prefilter:
    rng = np.random.default_rng(42)
    GEN_COLS = list(rng.choice(GEN_COLS, size=max_gen_prefilter, replace=False))

# Build float32 NumPy view (no copy if possible)
gen_arr = combined_df[GEN_COLS].to_numpy(dtype=np.float32, copy=False)

# Compute variance vectorized (NaN-safe)
gen_var = np.nanvar(gen_arr, axis=0)  # ddof=0 by default

# Keep columns above threshold (tune threshold)
gen_var_threshold = 1e-3  # genetics: drop mono/near-mono; raise to shrink more
gen_keep_mask = gen_var >= gen_var_threshold
GEN_COLS_RED = [c for c, keep in zip(GEN_COLS, gen_keep_mask) if keep]

print(f"GEN: {len(GEN_COLS)} → {len(GEN_COLS_RED)} (thr={gen_var_threshold})")

# ===== AROMA: simple (tiny set) =====
aroma_arr = combined_df[AROMA_COLS].to_numpy(dtype=np.float32, copy=False)
aroma_var = np.nanvar(aroma_arr, axis=0)
aroma_var_threshold = 1e-2  # typical for continuous peaks
aroma_keep_mask = aroma_var >= aroma_var_threshold
AROMA_COLS_RED = [c for c, keep in zip(AROMA_COLS, aroma_keep_mask) if keep]

print(f"AROMA: {len(AROMA_COLS)} → {len(AROMA_COLS_RED)} (thr={aroma_var_threshold})")

# ===== Build reduced matrices =====
X_reduced = combined_df[GEN_COLS_RED + AROMA_COLS_RED]
Y_all     = combined_df[TARGETS]

print("Reduced X:", X_reduced.shape, "| Y:", Y_all.shape)


Targets: 12 | GEN cols: 262434 | AROMA cols: 14
GEN: 30000 → 29838 (thr=0.001)
AROMA: 14 → 14 (thr=0.01)
Reduced X: (94, 29852) | Y: (94, 12)


Fast CV with supervised trim (SelectKBest inside Pipeline)

In [14]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
import numpy as np

cv = KFold(n_splits=3, shuffle=True, random_state=42)

GEN_COLS_RED   = [c for c in X_reduced.columns if c.startswith("GEN__")]
AROMA_COLS_RED = [c for c in X_reduced.columns if c.startswith("AROMA__")]
TARGETS        = [c for c in Y_all.columns if c.startswith("TARGET__")]

def score_model(X, y, pipe):
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="r2", n_jobs=-1)
    return float(scores.mean()), float(scores.std())

def make_rf():
    return RandomForestRegressor(
        n_estimators=300, max_depth=15, max_features="sqrt",
        max_samples=0.8, n_jobs=-1, random_state=42
    )

def pipes_for_target(idx, t, k_gen_grid=(0, 25, 50, 100, 200), gen_pcs=10):
    """Return X (both blocks), y (1-D for target t), and three pipelines (aroma, gen-pca, both[k])."""
    # X with both blocks
    X = X_reduced.loc[idx, AROMA_COLS_RED + GEN_COLS_RED]
    y = Y_all.loc[idx, t]                    # <-- 1-D Series for this target

    # AROMA-only
    pipe_aroma = Pipeline([
        ("ct", ColumnTransformer([
            ("aroma", Pipeline([("imp", SimpleImputer(strategy="median"))]), AROMA_COLS_RED),
        ], remainder="drop")),
        ("rf", make_rf())
    ])

    # GEN-only via PCA (safe wrt p>>n)
    n_pcs = min(gen_pcs, max(1, len(idx)-1))
    pipe_gen_pca = Pipeline([
        ("ct", ColumnTransformer([
            ("gen", Pipeline([("imp", SimpleImputer(strategy="median")),
                              ("pca", PCA(n_components=n_pcs, random_state=42))]), GEN_COLS_RED),
        ], remainder="drop")),
        ("rf", make_rf())
    ])

    # BOTH: keep all AROMA + add top-k GEN
    both_pipes = []
    for k in k_gen_grid:
        if k == 0:
            ct = ColumnTransformer([
                ("aroma", Pipeline([("imp", SimpleImputer(strategy="median"))]), AROMA_COLS_RED),
            ], remainder="drop")
        else:
            k_eff = min(k, len(GEN_COLS_RED))  # guard
            ct = ColumnTransformer([
                ("aroma", Pipeline([("imp", SimpleImputer(strategy="median"))]), AROMA_COLS_RED),
                ("gen",   Pipeline([("imp", SimpleImputer(strategy="median")),
                                    ("skb", SelectKBest(f_regression, k=k_eff))]), GEN_COLS_RED),
            ], remainder="drop")
        both_pipes.append( (k, Pipeline([("ct", ct), ("rf", make_rf())])) )

    return X, y, pipe_aroma, pipe_gen_pca, both_pipes


In [15]:
rows = []
for t in TARGETS:
    y_nonnull = Y_all[t].dropna()
    idx = y_nonnull.index

    X, y, pipe_aroma, pipe_gen_pca, both_pipes = pipes_for_target(idx, t)

    r2_aroma_m, r2_aroma_s = score_model(X.loc[:, AROMA_COLS_RED], y, pipe_aroma)
    r2_gen_m,   r2_gen_s   = score_model(X.loc[:, GEN_COLS_RED],   y, pipe_gen_pca)

    best_k, best_mean, best_sd = None, -1e9, 0.0
    for k, p in both_pipes:
        m, s = score_model(X, y, p)
        if m > best_mean:
            best_k, best_mean, best_sd = k, m, s

    rows.append({
        "target": t.replace("TARGET__",""),
        "AROMA_R2": r2_aroma_m, "AROMA_SD": r2_aroma_s,
        "GEN_PCA_R2": r2_gen_m, "GEN_PCA_SD": r2_gen_s,
        "BOTH_best_kGEN": best_k,
        "BOTH_R2": best_mean, "BOTH_SD": best_sd
    })

pd.DataFrame(rows).sort_values("BOTH_R2", ascending=False)


,target,AROMA_R2,AROMA_SD,GEN_PCA_R2,GEN_PCA_SD,BOTH_best_kGEN,BOTH_R2,BOTH_SD
5,Fresh Flavour,0.781264,0.070592,-0.133630,0.106846,0,0.781264,0.070592
7,Root/ Vegetable Flavour,0.774333,0.032124,-0.122675,0.140558,0,0.774333,0.032124
3,Earthy Flavour,0.761499,0.028788,-0.104780,0.140813,0,0.761499,0.028788
10,Sour Aftertaste,0.739993,0.071606,-0.149826,0.126398,0,0.739993,0.071606
2,Bitter Flavour,0.705320,0.067952,-0.135637,0.124843,0,0.705320,0.067952
1,Metallic Flavour,0.700355,0.060325,-0.089399,0.122358,0,0.700355,0.060325
9,Bitter Aftertaste,0.700005,0.032683,-0.083243,0.124556,0,0.700005,0.032683
4,Sour Flavour,0.688741,0.051163,-0.124521,0.127177,0,0.688741,0.051163
8,Farmyard (grass/hay) flavour,0.681942,0.061940,-0.133341,0.120629,0,0.681942,0.061940
6,Sweet Flavour,0.623133,0.047183,-0.041857,0.075865,0,0.623133,0.047183


<!-- Lasso regression -->

# Run Elastic Net & Random Forest on reduced dataset

In [ ]:


# --- 1) Feature selection once (using 'Sweet' as the proxy target) ---
k = min(500, X.shape[1])  # don't exceed number of features
selector = SelectKBest(score_func=f_regression, k=k)
selector.fit(X, y['Sweet'].fillna(0))

support = selector.get_support()
selected_cols = X.columns[support]

# Keep index + names so .loc works with your mask
X_selected = pd.DataFrame(
    selector.transform(X),
    index=X.index,
    columns=selected_cols
)

print(f"Shape after SelectKBest: {X_selected.shape}")

# --- 2) Models (scale for ElasticNet) ---
cv = KFold(n_splits=5, shuffle=True, random_state=42)
models = {
    'ElasticNet': make_pipeline(StandardScaler(), ElasticNet(max_iter=5000, random_state=42)),
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
}

# --- 3) Loop over flavors ---
for flavor in flavor_cols:
    mask = y[flavor].notna()            # rows with a label for this flavor
    X_train = X_selected.loc[mask]      # works now (index-aligned)
    y_train = y.loc[mask, flavor]

    print(f"\nResults for {flavor}:")
    for name, model in models.items():
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='r2')
        print(f"{name} CV R^2: {np.mean(scores):.3f} ± {np.std(scores):.3f}")


Shape after SelectKBest: (94, 500)

Results for Sweet:
ElasticNet CV R^2: 0.551 ± 0.072
RandomForest CV R^2: 0.782 ± 0.137

Results for Metallic Flavour:
ElasticNet CV R^2: 0.727 ± 0.084
RandomForest CV R^2: 0.886 ± 0.055

Results for Bitter Flavour:
ElasticNet CV R^2: 0.727 ± 0.104
RandomForest CV R^2: 0.862 ± 0.073

Results for Earthy Flavour:
ElasticNet CV R^2: 0.721 ± 0.054
RandomForest CV R^2: 0.931 ± 0.028

Results for Sour Flavour:
ElasticNet CV R^2: 0.704 ± 0.063
RandomForest CV R^2: 0.918 ± 0.036

Results for Fresh Flavour:
ElasticNet CV R^2: 0.778 ± 0.106
RandomForest CV R^2: 0.962 ± 0.018

Results for Sweet Flavour:
ElasticNet CV R^2: 0.665 ± 0.042
RandomForest CV R^2: 0.869 ± 0.020

Results for Root/ Vegetable Flavour:
ElasticNet CV R^2: 0.771 ± 0.053
RandomForest CV R^2: 0.958 ± 0.021

Results for Farmyard (grass/hay) flavour:
ElasticNet CV R^2: 0.706 ± 0.082
RandomForest CV R^2: 0.847 ± 0.080

Results for Bitter Aftertaste:
ElasticNet CV R^2: 0.731 ± 0.091
RandomForest CV

These results suggest strong predictability between your selected volatile features and flavor scores.

The 500 top features selected by SelectKBest seem to retain enough signal for both linear (ElasticNet) and non-linear (RandomForest) models.

High R² in CV (with low ±SD) suggests the model generalizes well — but 94 samples is still relatively small, so there’s always a risk of optimistic estimates if features are correlated.



In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

# 1) Remove "Intensity of Flavour" from X
X_no_intensity = X.drop(columns=["Intensity of Flavour"], errors="ignore")

cv = KFold(n_splits=5, shuffle=True, random_state=42)
k = min(200, X_no_intensity.shape[1])  # keep feature set modest for speed

for flavor in flavor_cols:
    mask = y[flavor].notna()
    Xf = X_no_intensity.loc[mask]
    yf = y.loc[mask, flavor]

    # --- Select top-k features ---
    selector = SelectKBest(score_func=f_regression, k=k)
    Xk = selector.fit_transform(Xf, yf)
    selected_cols = Xf.columns[selector.get_support()]

    # --- RandomForest ---
    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(Xk, yf)

    train_r2 = r2_score(yf, rf.predict(Xk))
    cv_r2 = cross_val_score(rf, Xk, yf, cv=cv, scoring="r2", n_jobs=-1)

    # --- Feature importances ---
    importances = pd.Series(rf.feature_importances_, index=selected_cols)
    top5 = importances.sort_values(ascending=False).head(5)

    print(f"\n=== {flavor} ===")
    print(f"Train R²: {train_r2:.3f} | CV R²: {np.mean(cv_r2):.3f} ± {np.std(cv_r2):.3f}")
    print("Top 5 features:\n", top5.to_string())



=== Sweet ===
Train R²: 0.945 | CV R²: 0.420 ± 0.418
Top 5 features:
 299105    0.179132
475518    0.076483
162964    0.040878
299077    0.039585
162999    0.039216

=== Metallic Flavour ===
Train R²: 0.911 | CV R²: 0.282 ± 0.120
Top 5 features:
 475481    0.076203
428792    0.064146
475551    0.046070
457062    0.045412
48675     0.042945

=== Bitter Flavour ===
Train R²: 0.911 | CV R²: 0.389 ± 0.100
Top 5 features:
 373299    0.042868
166707    0.038792
398375    0.033471
475481    0.033110
152264    0.032984

=== Earthy Flavour ===
Train R²: 0.913 | CV R²: 0.364 ± 0.118
Top 5 features:
 340351    0.042728
398375    0.037590
48675     0.031605
299275    0.030650
253306    0.029888

=== Sour Flavour ===
Train R²: 0.916 | CV R²: 0.382 ± 0.108
Top 5 features:
 475481    0.067544
475551    0.053889
475438    0.034340
260210    0.033766
398375    0.031870

=== Fresh Flavour ===
Train R²: 0.920 | CV R²: 0.412 ± 0.273
Top 5 features:
 149077    0.084851
152264    0.051922
162964    0.04623

 we see moderate predictive power — typical in sensory science, where flavor perception is multi-factorial and noisy.

The Train R² being much higher (≈0.91) than CV R² (≈0.35–0.45) suggests some overfitting is still happening — RandomForest is memorizing training data patterns that don’t generalize fully.



Top features are now actual volatile compound IDs (373299,166707,....)

<!-- Filter genetic markers by variance -->